In [ ]:
#pip install mlflow

In [ ]:
#pip install optuna

## ИМПОРТ БИБЛИОТЕК

In [1]:
import pandas as pd
import numpy as np
import re
import html
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
import mlflow
import mlflow.sklearn
import optuna
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
)

In [2]:
import kagglehub
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

Path to dataset files: /Users/ilya/.cache/kagglehub/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/versions/1


## НАЧАЛЬНЫЙ EDA И PREPROCESSING

In [3]:
df = pd.read_csv(path + '/IMDB Dataset.csv')

In [4]:
print(df.info())


<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     50000 non-null  str  
 1   sentiment  50000 non-null  str  
dtypes: str(2)
memory usage: 63.6 MB
None


In [5]:
df['sentiment'] = np.where(df['sentiment'] == 'positive', 1, 0)

In [6]:
print(df.head(5))


                                              review  sentiment
0  One of the other reviewers has mentioned that ...          1
1  A wonderful little production. <br /><br />The...          1
2  I thought this was a wonderful way to spend ti...          1
3  Basically there's a family where a little boy ...          0
4  Petter Mattei's "Love in the Time of Money" is...          1


In [7]:
with open("sample_review.txt", "w") as f:
    f.write(df['review'].iloc[1344])

print("Сохранено в sample_review.txt")

Сохранено в sample_review.txt


In [8]:
df['review_len'] = df['review'].str.split().str.len()

In [9]:
print(df['review_len'].max())

2470


In [10]:
df['review'] = df['review'].map(html.unescape)
df['review'] = df['review'].str.replace(r'<[^>]+>', '', regex=True)
df['review'] = df['review'].str.replace(r'[^a-z\s]', ' ', regex=True)
df['review'] = df['review'].str.replace(r'[\s+]', ' ', regex=True).str.strip()
print(df[df['review'].str.findall(r'<[^>]+>').explode().notna()]['review'])
df['review'] = df['review'].str.lower()

# pipeline обработки текста
# «Мы удаляем пунктуацию и цифры, потому что TF-IDF не работает с семантикой. В теории, ! и ? могли бы усиливать или менять смысл, но TF-IDF их не различает. Если бы мы использовали трансформеры (BERT), пунктуация была бы полезна — модель учитывает контекст. Для классического ML — удаление оправдано.»

Series([], Name: review, dtype: str)


In [11]:
# features_names = vectorizer.get_feature_names_out()
# coefs = model.coef_[0]

# positive_features = np.argsort(coefs)[-20:][::-1]
# negative_features = np.argsort(coefs)[:20]

# print('Positive:')
# for i in positive_features:
#   print(f'{features_names[i]} - {coefs[i]}')

# print('Negative:')
# for i in negative_features:
#   print(f'{features_names[i]} - {coefs[i]}')




In [12]:
print(df['review'].str.contains(r'\d+').sum())

0


In [13]:
df = df.drop('review_len', axis=1)

## TRAIN TEST SPLIT BALANCE

In [14]:

# Split train test
X = df['review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.8, random_state=42, stratify=y )

print(X_train)


16818    his was obviously the worst movie ever made   ...
45363    ame did something odd   t was not only a music...
8101     rom all the rave reviews  we couldn t wait to ...
23754    irst  let me confess that   have not read this...
3216     ickey  ourke is enjoying a renaissance at the ...
                               ...                        
42187    had high hopes when   went into the theatre   ...
22917    he first  and only  time   saw   hades  was du...
664      flying saucer manned  literally  by a crew of ...
11267    ife  tinks        was a step below  el  rooks ...
35597    this is by far the most pathetic movie  ndian ...
Name: review, Length: 10000, dtype: str


Я бы попробовал вариант б с перебором параметров друг под друга, но ограниченное кол-во для логистической регрессии. То есть мы дадим приоретет на разнообразие признаков, а модель проведем только через несколько вариантов, так как модель уже выдает близкие к максимальным значения, а значит тюнинг ее даст потенциально меньший прирост, чем хорошие данные


## MLFLOW tweak


In [16]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
experiment = client.get_experiment_by_name("IMDB Sentiment Analysis")

mlflow.set_experiment("IMDB Sentiment Analysis")

MlflowException: Cannot set a deleted experiment 'IMDB Sentiment Analysis' as the active experiment. You can restore the experiment, or permanently delete the experiment to create a new one.

## DISBALANCE FUNC

In [17]:
def make_disbalance(X, y, pos_ratio=0.1, random_state=42):
    X_train_imbalanced = X.reset_index(drop=True)
    y_train_imbalanced = y.reset_index(drop=True)

    pos_idx = y_train[y_train == 1].index
    neg_idx = y_train[y_train == 0].index
    
    n_neg = len(neg_idx)
    n_pos_keep = int(n_neg * pos_ratio / (1 - pos_ratio))
    
    rng = np.random.RandomState(random_state)
    pos_keep = rng.choice(pos_idx, size=n_pos_keep, replace=False)
    
    keep_idx = np.concatenate([pos_keep, neg_idx.values])
    rng.shuffle(keep_idx)
    
    return X_train.loc[keep_idx].reset_index(drop=True), y_train.loc[keep_idx].reset_index(drop=True)
    

## PIPELINE LogReg + OPTUNA

In [ ]:
def run_logreg(trial, balanced, C, max_iter, n_jobs, class_weight, min_df, max_df, sublinear_tf, ngram_range, max_features, random_seed):
    with mlflow.start_run(run_name='LogReg baseline'):

        mlflow.set_tag("optuna_trial", trial.number)
        mlflow.set_tag("model_name", "LogisticRegression")
        if balanced == 1:
            mlflow.set_tag("Balanced", "Balanced")
        else:
            mlflow.set_tag("Balanced", "Imbalanced")

        mlflow.log_param("C", C)
        mlflow.log_param("max_iter", max_iter)
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("class_weight", class_weight)


        model = LogisticRegression(
        C=C ,
        max_iter = max_iter,
        n_jobs=n_jobs,
        random_state=random_seed,
        class_weight=class_weight
        )

        vectorizer = TfidfVectorizer(min_df = min_df, max_df=max_df, sublinear_tf=sublinear_tf, ngram_range=ngram_range, max_features=max_features)
        if balanced == 1:
            y_trains = y_train
            X_train_tfidf = vectorizer.fit_transform(X_train)
            X_test_tfidf = vectorizer.transform(X_test)
        else:
            X_trains, y_trains = make_disbalance(X_train, y_train)
            X_train_tfidf = vectorizer.fit_transform(X_trains)
            X_test_tfidf = vectorizer.transform(X_test)



        mlflow.log_params({
                "tfidf_min_df": min_df,
                "tfidf_max_df": max_df,
                "tfidf_sublinear_tf": sublinear_tf,
                "tfidf_ngram_range": str(ngram_range),
                "tfidf_max_features": max_features,
                "tfidf_vocab_size": len(vectorizer.vocabulary_),
            })
        

        model.fit(X_train_tfidf, y_trains)

        y_pred = model.predict(X_test_tfidf)
        y_proba = model.predict_proba(X_test_tfidf)[:, 1]

        acc = accuracy_score(y_test, y_pred)

        f1_binary = f1_score(y_test, y_pred)
        f1_macro = f1_score(y_test, y_pred, average="macro")
        f1_weighted = f1_score(y_test, y_pred, average="weighted")

        prec_binary = precision_score(y_test, y_pred)
        rec_binary = recall_score(y_test, y_pred)

        prec_macro = precision_score(y_test, y_pred, average="macro")
        rec_macro = recall_score(y_test, y_pred, average="macro")
        
        roc_auc = roc_auc_score(y_test, y_proba)
        pr_auc = average_precision_score(y_test, y_proba)

        mlflow.log_metrics({
            "accuracy": acc,
            "f1": f1_binary,
            "f1_macro": f1_macro,
            "f1_weighted": f1_weighted,
            "precision": prec_binary,
            "recall": rec_binary,
            "precision_macro": prec_macro,
            "recall_macro": rec_macro,
            "roc_auc": roc_auc,
            "pr_auc": pr_auc,
        })

        mlflow.sklearn.log_model(model, "LogisticRegression")
    
    return f1_macro

In [ ]:

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
  print('Trialstarted')
  min_df = trial.suggest_categorical('min_df', [1, 2, 3, 5])
  max_df= trial.suggest_categorical('max_df', [0.85, 0.9, 0.95])
  sublinear_tf=trial.suggest_categorical('sublinear_tf', [True, False])
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
  max_features=trial.suggest_categorical('max_features', [50000, 60000, 40000, 100000])
  C=trial.suggest_categorical('C', [0.5, 0.8, 0.9, 0.95, 1, 1.1, 1.2, 2.0])
  max_iter = 1000
  n_jobs=-1
  class_weight=None
  balanced = 1
  random_seed=42
  
  return run_logreg(trial, balanced, C, max_iter, n_jobs, class_weight, min_df, max_df, sublinear_tf, ngram_range, max_features, random_seed)
  
study = optuna.create_study(direction='maximize', study_name='LogReg_TFIDF_optimization', sampler=optuna.samplers.TPESampler(seed=42))

study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f'\nBest F1: {study.best_value:.4f}')
print(f'\nParametrs: {study.best_params}')

## PIPELINE SVС + optuna

In [18]:
def run_svc(trial, balanced, C, loss, penalty, dual, max_iter, class_weight, min_df, max_df, sublinear_tf, ngram_range, max_features, random_seed):
    with mlflow.start_run(run_name=f'SVC trial_{trial.number}'):

        mlflow.set_tag("optuna_trial", trial.number)
        mlflow.set_tag("model_name", "SVC")
        if balanced == 1:
            mlflow.set_tag("Balanced", "Balanced")
        else:
            mlflow.set_tag("Balanced", "Imbalanced")

        mlflow.log_param("C", C)
        mlflow.log_param("max_iter", max_iter)
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("loss", loss)
        mlflow.log_param("penalty", penalty)
        mlflow.log_param("dual", dual)

        model = LinearSVC(
        C=C ,
        loss=loss,
        penalty=penalty,
        dual=dual, 
        max_iter = max_iter,
        class_weight=class_weight,
        random_state=random_seed
        )

        vectorizer = TfidfVectorizer(min_df = min_df, max_df=max_df, sublinear_tf=sublinear_tf, ngram_range=ngram_range, max_features=max_features)
        if balanced == 1:
            y_trains = y_train
            X_train_tfidf = vectorizer.fit_transform(X_train)
            X_test_tfidf = vectorizer.transform(X_test)
        else:
            X_trains, y_trains = make_disbalance(X_train, y_train)
            X_train_tfidf = vectorizer.fit_transform(X_trains)
            X_test_tfidf = vectorizer.transform(X_test)


        mlflow.log_params({
                "tfidf_min_df": min_df,
                "tfidf_max_df": max_df,
                "tfidf_sublinear_tf": sublinear_tf,
                "tfidf_ngram_range": str(ngram_range),
                "tfidf_max_features": max_features,
                "tfidf_vocab_size": len(vectorizer.vocabulary_),
            })

        model.fit(X_train_tfidf, y_trains)

        y_pred = model.predict(X_test_tfidf)
        y_proba = model.decision_function(X_test_tfidf)

        acc = accuracy_score(y_test, y_pred)

        f1_binary = f1_score(y_test, y_pred)
        f1_macro = f1_score(y_test, y_pred, average="macro")
        f1_weighted = f1_score(y_test, y_pred, average="weighted")

        prec_binary = precision_score(y_test, y_pred)
        rec_binary = recall_score(y_test, y_pred)

        prec_macro = precision_score(y_test, y_pred, average="macro")
        rec_macro = recall_score(y_test, y_pred, average="macro")
        
        roc_auc = roc_auc_score(y_test, y_proba)
        pr_auc = average_precision_score(y_test, y_proba)

        mlflow.log_metrics({
            "accuracy": acc,
            "f1": f1_binary,
            "f1_macro": f1_macro,
            "f1_weighted": f1_weighted,
            "precision": prec_binary,
            "recall": rec_binary,
            "precision_macro": prec_macro,
            "recall_macro": rec_macro,
            "roc_auc": roc_auc,
            "pr_auc": pr_auc,
        })

        mlflow.sklearn.log_model(model, "SVC")
        return f1_macro

In [ ]:
mlflow.set_experiment('IMDB Sentiment Analysis')
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_svc(trial):
  print('Trialstarted')
  min_df = trial.suggest_categorical('min_df', [1, 2, 3, 5])
  max_df= trial.suggest_categorical('max_df', [0.85, 0.9, 0.95])
  sublinear_tf=trial.suggest_categorical('sublinear_tf', [True, False])
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
  max_features=trial.suggest_categorical('max_features', [50000, 60000, 40000, 100000])
  C=trial.suggest_categorical('C', [0.5, 0.8, 0.9, 0.95, 1, 1.1, 1.2, 2.0])
  loss = trial.suggest_categorical('loss', ['squared_hinge'])
  penalty = trial.suggest_categorical('penalty', ['l2'])
  dual = trial.suggest_categorical('dual', [True])
  max_iter = 1000
  random_seed=42
  balanced=1
  class_weight=None
 
  return run_svc(trial, balanced, C, loss, penalty, dual, max_iter, class_weight, min_df, max_df, sublinear_tf, ngram_range, max_features, random_seed)

study_svc = optuna.create_study(direction='maximize', study_name='SVC_TFIDF_optimization', sampler=optuna.samplers.TPESampler(seed=42))

study_svc.optimize(objective_svc, n_trials=30, show_progress_bar=True)

print(f'\nBest F1: {study_svc.best_value:.4f}')
print(f'\nParametrs: {study_svc.best_params}')

## PIPELINE MULTINOMIAL NAIVE BIAS + OPTUNA

In [24]:
def run_MultinomialNB(trial, balanced, alpha, fit_prior, min_df, max_df, ngram_range, max_features):
    with mlflow.start_run(run_name=f'MultinomialNB trial_{trial.number}'):
        
        mlflow.set_tag("optuna_trial", trial.number)
        mlflow.set_tag("model_name", "MultinomialNB")
        if balanced == 1:
            mlflow.set_tag("Balanced", "Balanced")
        else:
            mlflow.set_tag("Balanced", "Imbalanced")

        mlflow.log_param("alpha", alpha)
        mlflow.log_param("fit_prior", fit_prior)

        model = MultinomialNB(
        fit_prior=fit_prior,
        alpha=alpha
        )
        
        vectorizer = CountVectorizer(min_df = min_df, max_df=max_df, ngram_range=ngram_range, max_features=max_features)
        if balanced == 1:
            y_trains = y_train
            X_train_tfidf = vectorizer.fit_transform(X_train)
            X_test_tfidf = vectorizer.transform(X_test)
        else:
            X_trains, y_trains = make_disbalance(X_train, y_train)
            X_train_tfidf = vectorizer.fit_transform(X_trains)
            X_test_tfidf = vectorizer.transform(X_test)

        mlflow.log_params({
                "min_df": min_df,
                "max_df": max_df,
                "ngram_range": str(ngram_range),
                "max_features": max_features,
                "vocab_size": len(vectorizer.vocabulary_),
            })

        model.fit(X_train_tfidf, y_trains)

        y_pred = model.predict(X_test_tfidf)
        y_proba = model.predict_proba(X_test_tfidf)[:, 1]

        acc = accuracy_score(y_test, y_pred)

        f1_binary = f1_score(y_test, y_pred)
        f1_macro = f1_score(y_test, y_pred, average="macro")
        f1_weighted = f1_score(y_test, y_pred, average="weighted")

        prec_binary = precision_score(y_test, y_pred)
        rec_binary = recall_score(y_test, y_pred)

        prec_macro = precision_score(y_test, y_pred, average="macro")
        rec_macro = recall_score(y_test, y_pred, average="macro")
        
        roc_auc = roc_auc_score(y_test, y_proba)
        pr_auc = average_precision_score(y_test, y_proba)

        mlflow.log_metrics({
            "accuracy": acc,
            "f1": f1_binary,
            "f1_macro": f1_macro,
            "f1_weighted": f1_weighted,
            "precision": prec_binary,
            "recall": rec_binary,
            "precision_macro": prec_macro,
            "recall_macro": rec_macro,
            "roc_auc": roc_auc,
            "pr_auc": pr_auc,
        })

        mlflow.sklearn.log_model(model, "MultinomialNB")
        return f1_macro


In [ ]:
mlflow.set_experiment('IMDB Sentiment Analysis')
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_MultiNB(trial):
  print('Trialstarted')
  min_df = trial.suggest_categorical('min_df', [1, 2, 3, 5])
  max_df= trial.suggest_categorical('max_df', [0.85, 0.9, 0.95])
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
  max_features=trial.suggest_categorical('max_features', [50000, 60000, 40000, 100000])
  alpha = trial.suggest_categorical('alpha', [0.1, 0.3, 0.5, 0.7, 0.9, 1.0, 2.0, 5.0, 7.0, 10.0])
  fit_prior = True
  balanced = 1

  return run_MultinomialNB(trial, balanced, alpha, fit_prior, min_df, max_df, ngram_range, max_features)
  
study_MultiNB = optuna.create_study(direction='maximize', study_name='MultinomialNB_optimization', sampler=optuna.samplers.TPESampler(seed=42))

study_MultiNB.optimize(objective_MultiNB, n_trials=30, show_progress_bar=True)

print(f'\nBest F1: {study_MultiNB.best_value:.4f}')
print(f'\nParametrs: {study_MultiNB.best_params}')

## DISBALANCE TEST

### LogReg

In [22]:

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_logreg_imbalanced(trial):
  print('Trialstarted')
  min_df = trial.suggest_categorical('min_df', [1, 2, 3, 5])
  max_df= trial.suggest_categorical('max_df', [0.85, 0.9, 0.95])
  sublinear_tf=trial.suggest_categorical('sublinear_tf', [True, False])
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
  max_features=trial.suggest_categorical('max_features', [50000, 60000, 40000, 100000])
  C=trial.suggest_categorical('C', [0.5, 0.8, 0.9, 0.95, 1, 1.1, 1.2, 2.0])
  max_iter = 1000
  n_jobs=-1
  class_weight=None
  random_seed=42
  balanced = 0
  
  return run_logreg(trial, balanced, C, max_iter, n_jobs, class_weight, min_df, max_df, sublinear_tf, ngram_range, max_features, random_seed)
  
study_logreg_imbalanced = optuna.create_study(direction='maximize', study_name='LogReg_TFIDF_optimization', sampler=optuna.samplers.TPESampler(seed=42))

study_logreg_imbalanced.optimize(objective_logreg_imbalanced, n_trials=30, show_progress_bar=True)

print(f'\nBest F1: {study_logreg_imbalanced.best_value:.4f}')
print(f'\nParametrs: {study_logreg_imbalanced.best_params}')

  0%|          | 0/30 [00:00<?, ?it/s]

Trialstarted


/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_44379/1422092892.py:8: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 2) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_44379/1422092892.py:8: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 3) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_44379/1422092892.py:8: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 1) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)]

Trialstarted


/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_44379/1422092892.py:8: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 2) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_44379/1422092892.py:8: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 3) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_44379/1422092892.py:8: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 1) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)]

...

Trialstarted


/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_44379/1422092892.py:8: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 2) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_44379/1422092892.py:8: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 3) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_44379/1422092892.py:8: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 1) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)]

Trialstarted


/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_44379/1422092892.py:8: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 2) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_44379/1422092892.py:8: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 3) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_44379/1422092892.py:8: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 1) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)]


Best F1: 0.4274

Parametrs: {'min_df': 5, 'max_df': 0.9, 'sublinear_tf': False, 'ngram_range': (1, 1), 'max_features': 50000, 'C': 2.0}


### SVC


In [22]:
mlflow.set_experiment('IMDB Sentiment Analysis')
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_svc_imbalanced(trial):
  print('Trialstarted')
  min_df = trial.suggest_categorical('min_df', [1, 2, 3, 5])
  max_df= trial.suggest_categorical('max_df', [0.85, 0.9, 0.95])
  sublinear_tf=trial.suggest_categorical('sublinear_tf', [True, False])
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
  max_features=trial.suggest_categorical('max_features', [50000, 60000, 40000, 100000])
  C=trial.suggest_categorical('C', [0.5, 0.8, 0.9, 0.95, 1, 1.1, 1.2, 2.0])
  loss = trial.suggest_categorical('loss', ['squared_hinge'])
  penalty = trial.suggest_categorical('penalty', ['l2'])
  dual = trial.suggest_categorical('dual', [True])
  max_iter = 1000
  random_seed=42
  class_weight=None
  balanced = 0

 
  return run_svc(trial, balanced,  C, loss, penalty, dual, max_iter, class_weight, min_df, max_df, sublinear_tf, ngram_range, max_features, random_seed)

study_svc_imbalanced = optuna.create_study(direction='maximize', study_name='SVC_TFIDF_optimization', sampler=optuna.samplers.TPESampler(seed=42))

study_svc_imbalanced.optimize(objective_svc_imbalanced, n_trials=30, show_progress_bar=True)

print(f'\nBest F1: {study_svc_imbalanced.best_value:.4f}')
print(f'\nParametrs: {study_svc_imbalanced.best_params}')

  0%|          | 0/30 [00:00<?, ?it/s]

Trialstarted


/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_69758/1796115475.py:9: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 2) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_69758/1796115475.py:9: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 3) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_69758/1796115475.py:9: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 1) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)]

Trialstarted


/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_69758/1796115475.py:9: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 2) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_69758/1796115475.py:9: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 3) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_69758/1796115475.py:9: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 1) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)]

...

Trialstarted


/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_69758/1796115475.py:9: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 2) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_69758/1796115475.py:9: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 3) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_69758/1796115475.py:9: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 1) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)]

Trialstarted


/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_69758/1796115475.py:9: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 2) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_69758/1796115475.py:9: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 3) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_69758/1796115475.py:9: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 1) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)]


Best F1: 0.6572

Parametrs: {'min_df': 5, 'max_df': 0.9, 'sublinear_tf': True, 'ngram_range': (1, 1), 'max_features': 50000, 'C': 2.0, 'loss': 'squared_hinge', 'penalty': 'l2', 'dual': True}


### MultinomialNB


In [ ]:
mlflow.set_experiment('IMDB Sentiment Analysis')
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_MultiNB_imbalanced(trial):
  print('Trialstarted')
  min_df = trial.suggest_categorical('min_df', [1, 2, 3, 5])
  max_df= trial.suggest_categorical('max_df', [0.85, 0.9, 0.95])
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
  max_features=trial.suggest_categorical('max_features', [50000, 60000, 40000, 100000])
  alpha = trial.suggest_categorical('alpha', [0.1, 0.3, 0.5, 0.7, 0.9, 1.0, 2.0, 5.0, 7.0, 10.0])
  fit_prior = True
  balanced = 0

  return run_MultinomialNB(trial, balanced, alpha, fit_prior, min_df, max_df, ngram_range, max_features)
  
study_MultiNB_imbalanced = optuna.create_study(direction='maximize', study_name='MultinomialNB_optimization', sampler=optuna.samplers.TPESampler(seed=42))

study_MultiNB_imbalanced.optimize(objective_MultiNB_imbalanced, n_trials=30, show_progress_bar=True)

print(f'\nBest F1: {study_MultiNB_imbalanced.best_value:.4f}')
print(f'\nParametrs: {study_MultiNB_imbalanced.best_params}')

  0%|          | 0/30 [00:00<?, ?it/s]

Trialstarted


/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_69758/1053818999.py:8: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 2) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_69758/1053818999.py:8: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 3) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_69758/1053818999.py:8: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 1) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)]

Trialstarted


/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_69758/1053818999.py:8: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 2) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_69758/1053818999.py:8: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 3) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_69758/1053818999.py:8: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 1) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)]

...

/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_69758/1053818999.py:8: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 2) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_69758/1053818999.py:8: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 3) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_69758/1053818999.py:8: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 1) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)]

Trialstarted


/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_69758/1053818999.py:8: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 2) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_69758/1053818999.py:8: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 3) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_69758/1053818999.py:8: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 1) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)]

Trialstarted


/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_69758/1053818999.py:8: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 2) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_69758/1053818999.py:8: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 3) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)])
/var/folders/l0/5l_ls00954978t3wqnx9tq_00000gn/T/ipykernel_69758/1053818999.py:8: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 1) which is of type tuple.
  ngram_range=trial.suggest_categorical('ngram_range', [(1, 2), (1, 3), (1, 1)]